In [49]:
import pandas as pd

df = pd.read_csv('./openings.csv')


In [50]:
import pandas as pd

def transform_chess_dataset(df, target_openings, n_continuations=3):
    # Create a master lookup from the original full dataset for move -> name
    # We use this to see if a sequence is "named" in the source data
    full_move_to_name = dict(zip(df['Moves'], df['Opening']))
    full_move_to_games = dict(zip(df['Moves'], df['Num Games']))

    def get_parent_move_string(move_str):
        if not move_str or move_str == "Start":
            return None
        parts = move_str.strip().split(' ')
        parts.pop()
        if parts and parts[-1].endswith('.'):
            parts.pop()
        return " ".join(parts) if parts else "Start"

    # --- 1 & 3. Identify Nodes (Targets + Ancestors) ---
    nodes_to_keep = set()
    for move_str in target_openings:
        parts = move_str.split(' ')
        for i in range(1, len(parts) + 1):
            prefix = " ".join(parts[:i])
            if not prefix.endswith('.'):
                nodes_to_keep.add(prefix)
    
    # --- 4. Identify Top N Continuations ---
    continuations = set()
    for target in target_openings:
        mask = df['Moves'].str.startswith(target + ' ')
        potential = df[mask].copy()
        potential['parent_check'] = potential['Moves'].apply(get_parent_move_string)
        top_n = (potential[potential['parent_check'] == target]
                 .sort_values('Num Games', ascending=False)
                 .head(n_continuations))
        continuations.update(top_n['Moves'].tolist())

    nodes_to_keep.update(continuations)

    # --- Recursive Name Resolver ---
    # This handles the "skipping" requirement
    memo_names = {"Start": "Starting Position"}
    
    def resolve_opening_name(move_str):
        if move_str in memo_names:
            return memo_names[move_str]
        
        # If it exists in the original dataset, use that name
        if move_str in full_move_to_name:
            res = full_move_to_name[move_str]
            memo_names[move_str] = res
            return res
        
        # If it doesn't exist, go back one step recursively
        parent_move = get_parent_move_string(move_str)
        parent_name = resolve_opening_name(parent_move)
        
        # Get the move suffix (e.g., "2. Nf3")
        if parent_move == "Start":
            suffix = move_str
        else:
            suffix = move_str[len(parent_move):].strip()
        
        res = f"{parent_name}, {suffix}"
        memo_names[move_str] = res
        return res

    # --- Build the Vertex Dataframe ---
    final_vertices = []
    
    # Process all identified nodes (plus "Start")
    all_nodes = list(nodes_to_keep) + ["Start"]
    for m in all_nodes:
        name = resolve_opening_name(m)
        games = full_move_to_games.get(m, 0) if m != "Start" else df['Num Games'].sum()
        
        final_vertices.append({
            'Opening': name,
            'Moves': m,
            'Num Games': games,
            'white_ideas': "", 'black_ideas': "", 
            'white_risks': "", 'black_risks': "", 'to_study': ""
        })

    v_df = pd.DataFrame(final_vertices).drop_duplicates(subset=['Moves'])

    # --- Create Edge Dataframe ---
    edges_data = []
    # Create local mapping for the current vertex set
    move_to_name_map = dict(zip(v_df['Moves'], v_df['Opening']))
    
    for _, row in v_df.iterrows():
        dst_move = row['Moves']
        dst_opening = row['Opening']
        
        if dst_move == "Start":
            continue
            
        src_move = get_parent_move_string(dst_move)
        
        if src_move in move_to_name_map:
            src_opening = move_to_name_map[src_move]
            
            # Formatting the move column
            if src_move == "Start":
                move_played = dst_move
            else:
                move_played = dst_move[len(src_move):].strip()
                # Handle black moves (e.g., "1...e5") for clarity in edges
                if '.' not in move_played:
                    # Find move number from the string
                    move_num = dst_move.rsplit('.', 1)[0].split(' ')[-1]
                    move_played = f"{move_num}...{move_played}"
            
            edges_data.append({
                'src': src_opening,
                'dst': dst_opening,
                'move': move_played
            })

    e_df = pd.DataFrame(edges_data)
    
    return v_df, e_df

In [51]:
targets = [
    "1.e4 e5 2.Nf3 Nc6 3.Bc4", # Italian
    "1.e4 e5 2.Nf3 Nc6 3.Bb5", # Spanish
    "1.e4 e5 2.Nf3 Nc6 3.Nc3 Nf6", # Four knights
    "1.e4 e5 2.Nf3 Nf6", # Russian
    "1.d4 d5 2.Nf3 Nf6", # Queen's pawn symmetrical
    "1.e4 d5 2.exd5", # Scandi
    "1.e4 e6", # French
    "1.e4 c6 2.d4 d5", # Caro
    "1.e4 c5" # Sicilian
]
vertices, edges = transform_chess_dataset(df, targets, n_continuations=5)

In [52]:
import os
v_cols = ['name', 'white_ideas', 'black_ideas', 'white_risks', 'black_risks', 'to_study', 'moves']

vertices = vertices.rename(columns={"Opening": "name", "Moves": "moves"})
edges = edges.rename(columns={"move": "added_moves"})

if os.path.exists("./vertices.csv"):
    old_vertices = pd.read_csv("./vertices.csv").rename(columns={"Opening": "name", "Moves": "moves"})
    old_edges = pd.read_csv("./edges.csv").rename(columns={"move": "added_moves"})
    
    vertices = pd.concat([old_vertices, vertices]).drop_duplicates("name").sort_values('name')
    edges = pd.concat([old_edges, edges]).drop_duplicates(["src", "dst"]).sort_values(['src', 'dst'])
    
vertices = vertices[v_cols]
    
# Check if there are existing vertices and edges. If so, just add the new rows
# I.e. drop duplicates on name and src


In [53]:
vertices.to_csv('vertices.csv', index=False)
edges.to_csv('edges.csv', index=False)

In [54]:
vertices.loc[vertices["name"] == "King Pawn Game, General, 2.Nf3"]

,name,white_ideas,black_ideas,white_risks,black_risks,to_study,moves
26,"King Pawn Game, General, 2.Nf3",NaN,NaN,NaN,NaN,NaN,1.e4 e5 2.Nf3
